# NoData, sessions portables et contrôle cartographique

**Cartomize · Version 1.0**

Durée indicative : 30 minutes.

## Objectifs

Préparer des copies masquées, protéger les valeurs valides, sauvegarder une carte et vérifier une révision.

## Apport de Cartomize

Les données sources sont conservées ; le projet garde les couches préparées, les paramètres, les références et les empreintes des fichiers.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("09_projets_controle")
d = demo(out/"data")

## 1. Examiner les couches et le fond

Un zéro peut être une classe valide. Une valeur noire à l’affichage n’est pas automatiquement NoData. L’analyse propose des règles ; leur application nécessite de connaître le produit.

In [3]:
background=cm.detect_background(d['before'],keep_values=(1,2,3))
display(background)
masked=cm.mask_background(d['before'],out/'copie_masquee.tif',keep_values=(1,2,3))
layers=[{'data':str(d['before']),'name':'Couverture','classes':CLASSES},str(d['points'])]
analysis=cm.analyze_project(layers,auto_background=False)
project=cm.prepare_project(layers,out/'projet',auto_background=False)
restored=cm.load_project(project.manifest)
original=cm.load_project(project.manifest,original_sources=True)
carte=restored.compose(title='Projet préparé',credits='Données synthétiques')
display(cm.analyze_relations([d['zones'],d['points']],metric_crs=CRS))

{'path': 'tutorials/outputs/09_projets_controle_okztp9vg/data/classes_t0.tif',
 'width': 128,
 'height': 96,
 'bands': 1,
 'bounds': [300000.0, 9497120.0, 303840.0, 9500000.0],
 'descriptions': ['synthetic_class'],
 'product': None,
 'valid_sample_pixels': 8593,
 'sample_values': [1.0, 2.0, 3.0],
 'sample_shape': [96, 128],
 'sampled_pixels': 12288,
 'crs': 'EPSG:32733',
 'dtype': 'uint8',
 'declared_nodata': [255.0],
 'mask_flags': [['nodata']],
 'candidates': [],
 'automatic_border_values': [],
 'binary_zero_preserved': False,
 'native_rgb': False,
 'ambiguous_values': [],
 'background_method': 'sampled_border_pattern',
 'decision_required': False,
 'note': 'Background inference is heuristic; sources and interior disconnected values remain unchanged.'}

{'schema': 'cartomize.relations.v1',
 'crs': 'EPSG:32733',
 'pairs': [{'left': 'zones',
   'right': 'points',
   'relation': 'contains',
   'distance_m': 0.0,
   'intersection_m2': 0.0,
   'scope': 'geometry'}],
 'note': 'Les rasters sont comparés par leur emprise ; les mesures sont planes dans la projection indiquée.'}

## 2. Sauvegarder les fichiers et les sessions

JSON conserve des références aux fichiers. CMZ peut inclure les données locales pour transporter le projet ; vérifier les droits de diffusion avant de partager une archive.

In [4]:
cm.save_map(carte,out/'carte.json')
reopened=cm.load_map(out/'carte.json')
carte.save(out/'carte.cmz',portable=True)
portable=cm.Map.load(out/'carte.cmz',directory=out/'extraction')
cm.save_session({'titre':'Exercice','sources':[str(d['image'])]},out/'session.json')
session=cm.load_session(out/'session.json')
assert len(portable.layers)==len(carte.layers)

## 3. Enregistrer une révision

Une empreinte détecte les modifications des données ou de la configuration. L’enregistrement suivant est un exemple pédagogique ; il ne représente pas une approbation scientifique de données réelles.

In [5]:
snapshot=cm.snapshot_project(carte)
unchanged=cm.compare_snapshots(snapshot,cm.snapshot_project(carte))
assert not unchanged['changed']
review=cm.record_review(snapshot,out/'revue.json',reviewer='Exercice pédagogique',comment='Contrôle de l’exemple synthétique')
assert cm.verify_review(review,snapshot)
carte.title='Titre modifié'
changed=cm.snapshot_project(carte)
assert not cm.verify_review(review,changed)
display(cm.compare_snapshots(snapshot,changed))

{'schema': 'cartomize.mapops.diff.v1',
 'changed': True,
 'settings_changed': True,
 'files': [],
 'previous': '48123f068e557de11536f0ef7d650d80dd0c15c4004e8c421fe55e34927daab3',
 'current': 'aa296d0d2518a644f3c3174b1b1e51e9264f20b0f31a93cdd37cd7d04a10b7b6'}

## 4. Passerelles QGIS et ArcGIS Pro

Cartomize fonctionne de manière autonome. `inspect_qgis_project` inventorie un QGS/QGZ ; `import_native_project` importe les éléments compatibles et rapporte les limites de transfert. `native_project` et `validate_native_runtime` exécutent des opérations dans un moteur installé. Ces opérations externes ne sont pas exécutées dans ce cours autonome. Voir [le guide des extensions et de la fenêtre](../DESKTOP_AND_EXTENSIONS.md).

## 5. Préserver un zéro valide pendant le masquage

Dans cet exemple, la valeur 254 représente un fond périphérique documenté et la valeur 0 une classe intérieure valide. Le masquage repose sur cette information, pas sur la couleur affichée.

In [6]:
from course import write_raster
values=np.full((64,64),254,dtype='uint8')
values[8:56,8:56]=1;values[24:40,24:40]=0
raw=write_raster(out/'fond_non_declare.tif',values)
clean=cm.mask_background(raw,out/'fond_corrige.tif',border_values=(254,),keep_values=(0,1))
with rasterio.open(clean["path"]) as src:
    data=src.read(1,masked=True)
    assert np.ma.getmaskarray(data)[0,:].all()
    assert not np.ma.getmaskarray(data)[24:40,24:40].any()
    assert (data[24:40,24:40]==0).all()
print('Fond périphérique masqué ; classe 0 conservée.')

Fond périphérique masqué ; classe 0 conservée.


## Exercice

Modifier seulement le titre puis vérifier la révision précédente. Pourquoi une nouvelle décision est-elle nécessaire ?

<details>
<summary>Éléments de correction</summary>

L’empreinte inclut les paramètres de carte. Une modification de titre change le document approuvé, même si les fichiers géographiques n’ont pas changé.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.